# Special Cases: Multiclass, Imbalanced Data, High-Cardinality Categoricals

Evaluation-safe usage note: `prepareXy` performs schema/type preparation only.
Discretization, HUG pattern mining, and downstream classifier fitting occur
inside `fit()` on the training data supplied to that call.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, balanced_accuracy_score
from hugiml import HUGIMLClassifierNative
import sys; from pathlib import Path; sys.path.insert(0, str(Path("../").resolve()))

## 1. Multiclass Classification (iris)

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
X_mc, y_mc = iris.data, iris.target.values
print(f"Iris: {X_mc.shape}, classes={np.unique(y_mc)}")

X_tr, X_te, y_tr, y_te = train_test_split(X_mc, y_mc, test_size=0.25,
                                            stratify=y_mc, random_state=42)
clf_mc = HUGIMLClassifierNative(B=5, L=2, G=1e-3)
clf_mc.fit(X_tr, y_tr)
preds = clf_mc.predict(X_te)
print(classification_report(y_te, preds, target_names=iris.target_names))

# Per-class pattern importances
from hugiml.multiclass import MulticlassHUGReport
report = MulticlassHUGReport(clf_mc)
print(report.summary(top_n=5))

Iris: (150, 4), classes=[0 1 2]
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        12
  versicolor       0.85      0.85      0.85        13
   virginica       0.85      0.85      0.85        13

    accuracy                           0.89        38
   macro avg       0.90      0.90      0.90        38
weighted avg       0.89      0.89      0.89        38

MulticlassHUGReport

Class: 0
----------------------------------------
  sepal width (cm)=[2,2.7)                  coef= -1.1901  sup=0.143
  sepal width (cm)=[3.4,4.4)                coef= +1.0913  sup=0.250
  sepal length (cm)=[4.3,5.02)              coef= +0.9612  sup=0.205
  petal width (cm)=[0.2,1.14)               coef= +0.8179  sup=0.357
  petal length (cm)=[1.1,1.5)               coef= +0.8112  sup=0.152

Class: 1
----------------------------------------
  petal length (cm)=[3.9,4.66)              coef= +1.2600  sup=0.205
  petal width (cm)=[1.14,1.5)               coef= +

## 2. Imbalanced Data (class_weight strategy)

In [ ]:
from sklearn.datasets import make_classification
from hugiml.multiclass import make_imbalanced_pipeline

X_imb, y_imb = make_classification(
    n_samples=2000, n_features=15, n_informative=8,
    weights=[0.9, 0.1], random_state=42
)
X_imb = pd.DataFrame(X_imb, columns=[f"f{i}" for i in range(X_imb.shape[1])])
print(f"Imbalanced: {X_imb.shape}, pos_rate={y_imb.mean():.3f}")

X_tr, X_te, y_tr, y_te = train_test_split(X_imb, y_imb, test_size=0.25,
                                            stratify=y_imb, random_state=42)

# Plain HUG-IML
clf_plain = HUGIMLClassifierNative(B=6, L=2, G=5e-3)
clf_plain.fit(X_tr, y_tr)
bal_plain = balanced_accuracy_score(y_te, clf_plain.predict(X_te))

# With class_weight='balanced'
clf_proto = HUGIMLClassifierNative(B=6, L=2, G=5e-3)
clf_bal = make_imbalanced_pipeline(clf_proto, strategy="class_weight")
clf_bal.fit(X_tr, y_tr)
bal_weighted = balanced_accuracy_score(y_te, clf_bal.predict(X_te))

print(f"Plain HUG-IML  balanced_acc={bal_plain:.4f}")
print(f"class_weight   balanced_acc={bal_weighted:.4f}")

Imbalanced: (2000, 15), pos_rate=0.100
Plain HUG-IML  balanced_acc=0.5628
class_weight   balanced_acc=0.6503


## 3. High-Cardinality Categoricals

In [ ]:
from hugiml.multiclass import encode_high_cardinality, apply_encoding

np.random.seed(42); n = 2000
cities = [f"city_{i:04d}" for i in range(500)]  # 500 unique cities
X_hc = pd.DataFrame({
    "city": np.random.choice(cities, n),
    "age":  np.random.randint(18, 70, n),
    "income": np.random.exponential(40000, n),
    "product": np.random.choice(["A","B","C","D","E"], n),
})
y_hc = ((X_hc["income"] > 40000) & (X_hc["age"] > 35)).astype(int).values

print(f"High-cardinality: {X_hc.shape}")
print(f"City cardinality: {X_hc['city'].nunique()}")

X_tr_raw, X_te_raw, y_tr, y_te = train_test_split(
    X_hc, y_hc, test_size=0.25, stratify=y_hc, random_state=42)

# Encode HIGH-cardinality column (city) with target-mean
X_tr_enc, enc_map = encode_high_cardinality(
    X_tr_raw, y_tr, threshold=10, method="target_mean")
X_te_enc = apply_encoding(X_te_raw, enc_map)

print(f"\nAfter encoding, 'city' dtype: {X_tr_enc['city'].dtype}")
print(f"Encoding map has {len(enc_map['city'])} city keys (fitted on train only)")

# Fit HUG-IML on encoded data
cat_cols = X_tr_enc.select_dtypes(include=["object","category"]).columns.tolist()
int_cols = X_tr_enc.select_dtypes(include="int").columns.tolist()
flt_cols = [c for c in X_tr_enc.columns if c not in cat_cols + int_cols]

clf_hc = HUGIMLClassifierNative(
    allCols=[int_cols, flt_cols, cat_cols],
    origColumns=X_tr_enc.columns.tolist(),
    B=6, L=2, G=5e-3,
)
clf_hc.fit(X_tr_enc, y_tr)
proba = clf_hc.predict_proba(X_te_enc)[:, 1]
print(f"\nHUG-IML on high-card data: ROC-AUC = {roc_auc_score(y_te, proba):.4f}")
print(clf_hc.model_summary())

High-cardinality: (2000, 4)
City cardinality: 500

After encoding, 'city' dtype: float64
Encoding map has 471 city keys (fitted on train only)

HUG-IML on high-card data: ROC-AUC = 0.9939
HUGIMLClassifierNative — Model Summary
Config:       B=6, L=2, G=0.005
Training:     1500 samples, 4 features, 2 classes
Patterns:     17 (0 compound)
Matrix:       (1500, 17) (density=0.1700)
Fit time:     156 ms

Stage breakdown (ms):
  resolve_meta                   5.4
  prepare_transactions         127.4
  mine_patterns                  4.6
  build_matrix                   0.8
  fit_downstream                 5.8
  compat                         0.3

Top 10 patterns by importance:
  age=[27,35)                              coef= -3.9771  sup=0.162
  age=[18,27)                              coef= -3.7799  sup=0.163
  income=[6.786e+04,2.926e+05)             coef= +3.5692  sup=0.167
  income=[4.223e+04,6.786e+04)             coef= +3.4767  sup=0.167
  income=[6822,1.614e+04)                  coef= 

## 4. Adaptive Binning

In [ ]:
from hugiml.adaptive import HUGIMLAdaptive

clf_adapt = HUGIMLAdaptive(b_candidates=[3,5,7,10,15], L=2, G=5e-3)
clf_adapt.prepareXy(X_hc, y_hc)
clf_adapt.fit(X_tr_enc, y_tr)
proba_adapt = clf_adapt.predict_proba(X_te_enc)[:, 1]
print(f"Adaptive HUG-IML: ROC-AUC = {roc_auc_score(y_te, proba_adapt):.4f}")
print("Per-feature chosen B:")
for feat, b in sorted(clf_adapt.per_feature_b_.items()):
    print(f"  {feat:<20} B={b}")

High-cardinality: (2000, 4)
City cardinality: 500

After encoding, 'city' dtype: float64
Encoding map has 471 city keys (fitted on train only)

HUG-IML on high-card data: ROC-AUC = 0.9939
HUGIMLClassifierNative — Model Summary
Config:       B=6, L=2, G=0.005
Training:     1500 samples, 4 features, 2 classes
Patterns:     17 (0 compound)
Matrix:       (1500, 17) (density=0.1700)
Fit time:     156 ms

Stage breakdown (ms):
  resolve_meta                   5.4
  prepare_transactions         127.4
  mine_patterns                  4.6
  build_matrix                   0.8
  fit_downstream                 5.8
  compat                         0.3

Top 10 patterns by importance:
  age=[27,35)                              coef= -3.9771  sup=0.162
  age=[18,27)                              coef= -3.7799  sup=0.163
  income=[6.786e+04,2.926e+05)             coef= +3.5692  sup=0.167
  income=[4.223e+04,6.786e+04)             coef= +3.4767  sup=0.167
  income=[6822,1.614e+04)                  coef= 

## 5. Pattern Pruning (regulated editing workflow)

In [ ]:
from hugiml.pruning import PatternEditor

editor = PatternEditor(clf_hc, operator_name="data_scientist")
print(f"Before: {editor.diff()['n_original']} patterns")

# Remove patterns with support < 2%
editor.remove_low_support(min_support=0.02, reason="low support — unstable in production")
print(f"After low-support removal: {editor.diff()['n_current']} patterns")

# Refit downstream classifier
editor.refit(X_tr_enc, y_tr)

# Calibrate probabilities on a held-out calibration set
editor.calibrate(X_te_enc, y_te, method="isotonic")

# Export to final model
new_clf = editor.finalize()
print(editor.audit_report())

proba_pruned = new_clf.predict_proba(X_te_enc)[:, 1]
print(f"Pruned model: ROC-AUC = {roc_auc_score(y_te, proba_pruned):.4f}")

Before: 17 patterns
After low-support removal: 17 patterns
{
  "operator": "data_scientist",
  "generated_at": "2026-05-22T13:31:47.528251",
  "diff": {
    "n_original": 17,
    "n_current": 17,
    "n_removed": 0,
    "removed_patterns": []
  },
  "calibration": {
    "applied": false,
    "method": null
  },
  "removals": []
}
Pruned model: ROC-AUC = 0.9939
